In [ ]:
import cv2 as cv # openCV 
from pathlib import Path
import numpy as np
import pandas as pd

kernel_size = 9
file_names = []

for file in Path("../data/testing/board_images").iterdir(): 
  if (file.name.endswith(".jpeg")):
     file_names.append(file.name)
  
for file in file_names:
  image = cv.imread(f"../data/testing/board_images/{file}")
  grayscale = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
  blur = cv.GaussianBlur(grayscale, (kernel_size, kernel_size), 3)

  # params: image (post gaussian blur), lower threshold, upper threshold
  # upper threshold = minimum required to be an edge
  # low threshold = minimum required to be an edge, assuming it touches an existing edge (defined by high threshold)
  edges = cv.Canny(blur, 50, 100)

  # params (don't touch first 3)
  # threshold = min number of votes to draw line
  # minLineLength = get rid of segments shorter than that
  # maxLineGap = max gap between lines to be merged into a line
  lines = cv.HoughLinesP(edges, 1, np.pi / 180, threshold=100, minLineLength=20, maxLineGap=150)

  if lines is not None:
      for line in lines:
          x1, y1, x2, y2 = line
          cv.line(image, (x1, y1), (x2, y2), (255, 0, 255), 3)

  cv.imwrite(f"../data/testing/processed_{file}", image)


In [ ]:
import cv2 as cv 
import numpy as np 
from sklearn.cluster import DBSCAN
import heapq
from utils import draw_points, write_image, find_line_intersection, is_kinda_vertical, distance_to
import pandas as pd

image = cv.imread("../data/testing/board_images/empty_board.jpeg")
blur = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
blur = cv.GaussianBlur(blur, (7, 7), 3)
edges = cv.Canny(blur, 50, 100)
lines = cv.HoughLinesP(edges, 1, np.pi / 180, threshold=75, minLineLength=20, maxLineGap=150)
height = 3500
width = 3000
corners = []

black_img = np.zeros((height, width), np.uint8)

horizontal_lines = []
vertical_lines = []


if lines is not None: 
   for line in lines: 
      if is_kinda_vertical(line): 
        vertical_lines.append(line)
      else: 
        horizontal_lines.append(line)

horizontal = False
vertical = False

if horizontal_lines is not None and horizontal:
    for line in horizontal_lines:
        x1, y1, x2, y2 = line
        cv.line(black_img, (x1, y1), (x2, y2), (255, 0, 255), 1)

if vertical_lines is not None and vertical: 
   for line in vertical_lines: 
      x1, y1, x2, y2 = line
      cv.line(black_img, (x1, y1), (x2, y2), (255, 0, 255), 1)

# Finding all intersections between the vertical and horizontal groups of lines 
# then clustering the points together 
for line in vertical_lines:
   for line2 in horizontal_lines: 
      x, y = find_line_intersection(line, line2)
      if (x != 0 and y!= 0):
        corners.append((round(x),round(y))) 
pts = np.asarray(corners)
labels = DBSCAN(eps=20, min_samples=3).fit_predict(pts)
clustered_corners = np.array([pts[labels == k].mean(axis=0) for k in np.unique(labels)])

# plan from here: 
# for each point, find the distance to the closest four points 
# first do a check to make sure the points are NOT diagonal (not too strict though)
# make a big old list 
# find IQR, get rid of outliers 
# only keep points that have at least two buddies within the range

vertical_horizontal_threshold = 25 # defines how far away in the x or y direction two points can be to still be considered "vertical" or "horizontal" 
distance = []
for point_1 in clustered_corners: 
  for point_2 in clustered_corners:

    # filter to only check points that are more or less vertical / horizontal
    if (abs(point_1[0] - point_2[0]) < vertical_horizontal_threshold or abs(point_1[1] - point_2[1]) < vertical_horizontal_threshold):
      if (point_1[0] != point_2[0] and point_1[1] != point_2[1]):
        distance.append(distance_to(point_1, point_2))
      
distance = pd.DataFrame({"distance": distance})
distance = distance['distance'].sort_values().reset_index(drop=True)

distance_between_points_threshold = 30 # use smaller when there's little to no warping. larger accounts for some warping in the photo
left = 0
right = 0 
max_difference = 0

# finds the most common difference between points by using a sliding window through distance, 
# where distance is a list of the distance to the closest 4 neighboring points (for each point). len(distance) = n * 4, where n = # of points 
while (left < len(distance) and right < len(distance)): 
  if (distance[left] + distance_between_points_threshold > distance[right]): 
    right += 1 
  else: 
    left += 1
  if right - left > max_difference: 
    max_difference = right - left 
    most_common_dist = distance[right - (max_difference // 2)]

filtered_corners = []
distance_threshold = 15
black_img = np.zeros((3500, 3500, 3), dtype=np.uint8)

# filtering the corners by how many good neighbors they have 

# adds a point to filtered_corners if at least two neighbors are within most_common_distance +/- the buffer 
for point_1 in clustered_corners:
  num_times = 0
  for point_2 in clustered_corners: 
    if (abs(point_1[0] - point_2[0]) < vertical_horizontal_threshold or abs(point_1[1] - point_2[1]) < vertical_horizontal_threshold):
      if (point_1[0] != point_2[0] and point_1[1] != point_2[1]):
        if distance_to(point_1, point_2) >= most_common_dist - distance_threshold:
          if distance_to(point_1, point_2) <= most_common_dist + distance_threshold:
            num_times += 1
            if num_times == 2: 
              filtered_corners.append(point_1)


image = cv.imread("../data/testing/board_images/empty_board.jpeg")
draw_points(black_img, filtered_corners) 

cv.line(black_img, (int(filtered_corners[15][0]), int(filtered_corners[15][1])), (int(filtered_corners[15][0] + most_common_dist), int(filtered_corners[15][1])), (255, 255, 0), 10) 

draw_points(image, filtered_corners) 
write_image(black_img, "black_background")
write_image(image, "filtered_corners")

# scoring corners based on distance to 4 nearest neighbors

# Running into an issuue where an outliter point has a neighbor 0.5x most_common_dist and 1.5x most_common_dist away... 
# To solve this, i am planning on ignoring neighbors that are closer than 0.7x most_common_dist together, but also dropping MAX_CLOSEST_NEIGHBORS to 3. 
# This will score those points on only the 3 other closest neighbors, resulting in the point that has a 1.5x neighbor to get scored worse 


# second issue is that this is very susceptible to losing the points that are located one edges (because they have at most 3 neighbors) 
# Really don't want to lose edge corners... 
# Fixes: 
# Could look at neighbors two lengths away 

corner_score = []

for x, point_1 in enumerate(filtered_corners): 
  closest_four_distances = []
  MAX_CLOSEST_NEIGHBORS = 4
  for y, point_2 in enumerate(filtered_corners):
    if x != y: 
       # filter to only check points that are more or less vertical / horizontal
      if (abs(point_1[0] - point_2[0]) < vertical_horizontal_threshold or abs(point_1[1] - point_2[1]) < vertical_horizontal_threshold):
        distance_to_neighbor = distance_to(point_1, point_2) 
        if distance_to_neighbor < 0.7 * most_common_dist:
          MAX_CLOSEST_NEIGHBORS -= 1 if MAX_CLOSEST_NEIGHBORS > 1 else 1
          continue
        if len(closest_four_distances) < MAX_CLOSEST_NEIGHBORS: 
          # negating the distance before adding it to the heap, because it's a max heap (and I want a min heap) 
          heapq.heappush(closest_four_distances, -distance_to_neighbor)
        else: 
          heapq.heappushpop(closest_four_distances, -distance_to_neighbor)

  # taking the average length of the four closest neighbors, subtracted by most_common_dist 
  while len(closest_four_distances) > MAX_CLOSEST_NEIGHBORS: 
    heapq.heappop(closest_four_distances)

  avg_dist = sum(closest_four_distances) / len(closest_four_distances) 
  heapq.heappush(corner_score, (abs(abs(avg_dist) - most_common_dist), point_1))
  
  # values near zero are good, values away from zero are bad

score, point = heapq.heappop(corner_score)
verified_points = [point] 

while score < 10: 
  # cv.circle(image, (int(point[0]), int(point[1])), 15, (0, 0, 255), -1)
  score, point = heapq.heappop(corner_score)
  verified_points.append(point)
  # print(score)

image = cv.imread("../data/testing/board_images/empty_board.jpeg")
draw_points(image, verified_points)
write_image(image, "verified")

closeness_to_verified_point = 10 

for _ in range (2):
  already_added = []
  verified_and_buddies = verified_points.copy()

  for point_1 in verified_points:
    for index, point_2 in enumerate(filtered_corners):
      if index not in already_added: 
        if abs(point_1[0] - point_2[0]) > most_common_dist - closeness_to_verified_point and abs(point_1[0] - point_2[0]) < most_common_dist + closeness_to_verified_point:
          if abs(point_1[1] - point_2[1]) < vertical_horizontal_threshold: 
            verified_and_buddies.append(point_2)
            already_added.append(index)
        elif abs(point_1[1] - point_2[1]) > most_common_dist - closeness_to_verified_point and abs(point_1[1] - point_2[1]) < most_common_dist + closeness_to_verified_point: 
          if abs(point_1[0] - point_2[0]) < vertical_horizontal_threshold: 
            verified_and_buddies.append(point_2)
            already_added.append(index)
  verified_points = verified_and_buddies.copy()
 

image = cv.imread("../data/testing/board_images/empty_board.jpeg")

draw_points(image, verified_and_buddies)
write_image(image, "verified_and_buddies")

from chessboard import Chessboard

test = Chessboard(most_common_dist)

image = cv.imread("../data/testing/board_images/empty_board.jpeg")
test.add_points(verified_and_buddies)
test.imputate_points()
corners = test.get_corners()
draw_points(image, corners)
write_image(image, "CORNERS") 

